<a href="https://colab.research.google.com/github/sstanishk/tanishk-codeboosters-2026/blob/main/Day3/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully')
print(f'pandas : {pd.__version__}')
print(f'numpy : {np.__version__}')
print(f'requests : {requests.__version__}')


All libraries imported successfully
pandas : 2.2.2
numpy : 2.0.2
requests : 2.32.4


In [44]:
raw_df = pd.read_csv('messy_sales_data (1).csv')
print(f'Raw data loaded: {raw_df.shape[0]} rows, {raw_df.shape[1]}columns')
print(raw_df.head())

Raw data loaded: 32 rows, 9columns
   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       NaN       22000   
4      1005  Ramesh Kumar    Laptop  Electronics       2.0       45000   

   order_date       city    sales_rep  
0  05-01-2024     Mumbai  Anil Sharma  
1  07-01-2024      Delhi   Sunita Rao  
2  08-01-2024  Bangalore  Anil Sharma  
3  10-01-2024    Chennai   Ravi Kumar  
4  05-01-2024     Mumbai  Anil Sharma  


In [45]:
print('='*55)
print(' DATA QUALITY DIAGNOSIS REPORT')
print('='*55)

print('\n[1] MISSING VALUES PER COLUMN')
print(raw_df.isnull().sum())

print('\n[2] DUPLICATED ROWS')
print(f'There are {raw_df.duplicated().sum()} duplicated rows')

print('\n[3] DATA TYPES:')
print(raw_df.dtypes)

print('\n[4] UNIQUE CATEGORIES:',raw_df['category'].unique())
print('\n[4] sample customer names:',raw_df['customer_name'].dropna().unique()[:8])
print('\n[4] sample order_date:',raw_df['order_date'].dropna().unique()[:6])

 DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALUES PER COLUMN
order_id         0
customer_name    2
product          1
category         1
quantity         4
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATED ROWS
There are 2 duplicated rows

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]

[4] sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']

[4] sample order_date: ['05-01-2024' '07-01-2024' '08-01-2024' '10-01-2024' '12-01-2024'
 '13-01-2024']


In [46]:
df = raw_df.copy()
print(f'working copy created: {df.shape}')
print(f'raw_df is untouched - we can always reset by running df =  ')


working copy created: (32, 9)
raw_df is untouched - we can always reset by running df =  


In [51]:
print('Before fixing nulls:', df.isnull().sum().sum(), 'total missing')
df['customer_name'].fillna('Unknown Customer', inplace=True)

median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f"Filled missing quantity with median: {median_qty}")

df['category'].fillna('Uncategorized', inplace=True)
df['product'].fillna('Unknown product', inplace=True)
print('After fixing nulls:', df.isnull().sum().sum(), 'total missing values')



Before fixing nulls: 0 total missing
Filled missing quantity with median: 2.5
After fixing nulls: 0 total missing values


In [52]:
# how to remove duplicates
print(f'Before deduplication:{len(df)}rows')
print(f'before deduplication:{df.duplicated().sum()}duplicates')

df.drop_duplicates(inplace=True)
print(f'after deduplication:{len(df)}rows')
print(f'after deduplication:{df.duplicated().sum()}duplicates')

Before deduplication:30rows
before deduplication:0duplicates
after deduplication:30rows
after deduplication:0duplicates


In [54]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())

df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=False, errors='coerce')

nat_count = df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')

df['year'] = df['order_date'].dt.year

df['month'] = df['order_date'].dt.month

df['month_name'] = df['order_date'].dt.strftime('%B')

print('\nSample dates after parsing:')
print(df[['order_date', 'year', 'month', 'month_name']].head(5))

Sample dates before parsing:
['05-01-2024', '07-01-2024', '08-01-2024', '10-01-2024', '05-01-2024', '07-01-2024', '12-01-2024', '13-01-2024']

Unparseable dates (NaT): 16

Sample dates after parsing:
  order_date    year  month month_name
0 2024-05-01  2024.0    5.0        May
1 2024-07-01  2024.0    7.0       July
2 2024-08-01  2024.0    8.0     August
3 2024-10-01  2024.0   10.0    October
4 2024-05-01  2024.0    5.0        May


In [55]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())

df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')

nat_count = df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')

# Extract year and month, which will result in NaN for NaT dates.
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month

# Fill NaN values in 'year' and 'month' with 0 before converting to integer.
df['year'] = df['year'].fillna(0).astype(int)
df['month'] = df['month'].fillna(0).astype(int)

df['month_name'] = df['order_date'].dt.strftime('%B')
# Fill NaN values in 'month_name' with 'Unknown'
df['month_name'].fillna('Unknown', inplace=True)

# order_date format change - this line should be after year/month extraction
# Convert datetime objects to string format, and handle 'NaT' strings.
df['order_date'] = df['order_date'].dt.strftime('%d-%m-%Y').replace({'NaT': 'Unknown Date'})

print('\nSample dates after parsing:')
print(df[['order_date', 'year', 'month', 'month_name']].head(5))

Sample dates before parsing:
[Timestamp('2024-05-01 00:00:00'), Timestamp('2024-07-01 00:00:00'), Timestamp('2024-08-01 00:00:00'), Timestamp('2024-10-01 00:00:00'), Timestamp('2024-05-01 00:00:00'), Timestamp('2024-07-01 00:00:00'), Timestamp('2024-12-01 00:00:00'), NaT]

Unparseable dates (NaT): 16

Sample dates after parsing:
   order_date  year  month month_name
0  01-05-2024  2024      5        May
1  01-07-2024  2024      7       July
2  01-08-2024  2024      8     August
3  01-10-2024  2024     10    October
4  01-05-2024  2024      5        May
